## did it too fast so most likely sucks but we'll see:

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve, svds
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import cma

# ==========================================
# 1. DATA MODELS: Components and Constraints
# ==========================================

class RigidComponent:
    def __init__(self, comp_id, initial_state):
        """
        initial_state: [alpha, x, y] representing rotation and translation
        """
        self.id = comp_id
        # State over time [n_samples, 3]
        self.states = np.array([initial_state]) 
        
    def get_global_point(self, local_u, local_v, time_idx=0):
        """Converts local coordinates (u,v) to world space at a given time step."""
        state = self.states[time_idx]
        alpha, x, y = state
        
        # 2D Rotation matrix
        R = np.array([
            [np.cos(alpha), -np.sin(alpha)],
            [np.sin(alpha),  np.cos(alpha)]
        ])
        local_vec = np.array([local_u, local_v])
        return R.dot(local_vec) + np.array([x, y])

class Assembly:
    def __init__(self):
        self.components = []
        self.constraints = []
        
    def add_component(self, comp):
        self.components.append(comp)

# ==========================================
# 2. CORE MATHEMATICAL KERNEL: Kinematics
# ==========================================

class ForwardKinematicsSolver:
    def __init__(self, assembly, n_steps=100):
        self.assembly = assembly
        self.n_steps = n_steps

    def compute_jacobian(self, global_state):
        """
        Constructs the sparse Jacobian matrix J of the constraints.
        For a real implementation, you derive dC/ds for Pin, Motor, and Ground constraints.
        Here we mock a structurally valid sparse matrix for SVD testing.
        """
        n_vars = len(self.assembly.components) * 3
        n_constraints = n_vars # Assuming fully constrained system
        
        # Mocking a block diagonal Jacobian for structural representation
        J = sp.diags([1, -1, 0.5], offsets=[0, 1, -1], shape=(n_constraints, n_vars), format='csr')
        return J
        
    def get_smallest_singular_value(self, global_state):
        """Calculates lambda_min using sparse SVD to detect singularities."""
        J = self.compute_jacobian(global_state)
        # svds is highly efficient for large sparse matrices when only extreme values are needed
        try:
            # k=1 (one singular value), which='SM' (smallest magnitude)
            u, s, vt = svds(J, k=1, which='SM')
            return s[0]
        except:
            # Fallback if matrix is perfectly singular and svds fails to converge
            return 1e-6 

# ==========================================
# 3. TOPOLOGY DESIGN: Linkage Placement
# ==========================================

class TopologyOptimizer:
    def __init__(self, comp_a, comp_b, comp_motor, time_series_states):
        self.ca = comp_a
        self.cb = comp_b
        self.cm = comp_motor
        self.n_samples = len(time_series_states)
        self.gamma = 0.1 # Weighting factor for moment arm penalty
        
    def objective_function(self, params):
        """
        params: [ua, va, ub, vb]
        Calculates distance variance \delta_{ab} + area penalty E_area
        """
        ua, va, ub, vb = params
        
        xa_t = []
        xb_t = []
        xm_t = [] # Motor positions
        
        # Reconstruct world positions over time
        for i in range(self.n_samples):
            xa_t.append(self.ca.get_global_point(ua, va, i))
            xb_t.append(self.cb.get_global_point(ub, vb, i))
            # Assuming motor is at origin (0,0) of comp_motor
            xm_t.append(self.cm.get_global_point(0, 0, i)) 
            
        xa_t = np.array(xa_t)
        xb_t = np.array(xb_t)
        xm_t = np.array(xm_t)
        
        # 1. Variance Calculation (delta_ab)
        distances = np.linalg.norm(xa_t - xb_t, axis=1)
        l_ab = np.mean(distances**2)
        variance_penalty = np.mean((distances**2 - l_ab)**2)
        
        # 2. Area Calculation (Moment Arm Penalty)
        # area = 0.5 * |(xb - xa) cross (xm - xa)|
        vec1 = xb_t - xa_t
        vec2 = xm_t - xa_t
        areas = 0.5 * np.abs(vec1[:, 0]*vec2[:, 1] - vec1[:, 1]*vec2[:, 0])
        
        # Avoid log(0)
        area_sum_sq = np.sum(areas**2) + 1e-8
        area_penalty = -np.log(area_sum_sq)
        
        return variance_penalty + self.gamma * area_penalty

    def optimize_multi_start(self, n_restarts=5):
        best_params = None
        best_cost = np.inf
        
        for _ in range(n_restarts):
            initial_guess = np.random.uniform(-5, 5, 4)
            res = minimize(self.objective_function, initial_guess, method='BFGS')
            if res.fun < best_cost:
                best_cost = res.fun
                best_params = res.x
                
        return best_params

# ==========================================
# 4. GLOBAL OPTIMIZATION: Fine Tuning (CMA-ES)
# ==========================================

def global_kinematic_optimization(initial_params, solver, target_markers):
    """
    Minimizes: w1*E_marker + w2*E_state + w3*E_joint + w4*E_singular
    Using Derivative-Free CMA-ES.
    """
    w1, w2, w3, w4 = 1.0, 0.5, 0.1, 10.0
    epsilon = 1e-4
    alpha = 2.0
    
    def cost_function(params):
        # 1. Apply params to assembly (mocked here)
        cost = 0.0
        
        # 2. Simulate forward kinematics to get s(t)
        # s_t = solver.solve_full_cycle(params)
        
        # 3. Calculate Singular Penalty across time cycle
        # We aggressively penalize rank-deficient matrices
        e_singular = 0
        for i in range(solver.n_steps):
            lambda_min = solver.get_smallest_singular_value(None) # Pass state here
            e_singular += 1.0 / ((lambda_min + epsilon)**alpha)
            
        cost += w4 * e_singular
        
        # ... (Add E_marker, E_state, E_joint logic based on generated s_t)
        return cost

    # Initialize CMA-ES with initial standard deviation of 0.5
    es = cma.CMAEvolutionStrategy(initial_params, 0.5)
    
    # Normally you would use es.optimize(cost_function), keeping it brief for the visualizer
    print("Initialized CMA-ES for global refinement. Avoiding execution in UI block.")
    return initial_params 


# ==========================================
# 5. JUPYTER VISUALIZER: Interactive Kinematics
# ==========================================

def launch_jupyter_visualizer():
    """
    Creates an interactive slider for JupyterLab to scrub through the linkage motion.
    To make this visually compelling out-of-the-box, we generate a synthetic 
    4-bar linkage (Crank-Rocker) dataset to demonstrate the plotting methodology.
    """
    # --- Generate Synthetic Kinematic Data for Visualization ---
    n_frames = 100
    theta = np.linspace(0, 2*np.pi, n_frames)
    
    # Ground pins
    p0 = np.array([0, 0])
    p3 = np.array([4, 0])
    
    # Link lengths
    L1, L2, L3 = 1.5, 5.0, 4.0
    
    frames_p1 = []
    frames_p2 = []
    frames_eff = [] # End effector path
    
    for t in theta:
        # Crank position
        p1 = p0 + np.array([L1*np.cos(t), L1*np.sin(t)])
        
        # Solve for p2 (intersection of two circles from p1 and p3)
        d = np.linalg.norm(p3 - p1)
        a = (L2**2 - L3**2 + d**2) / (2*d)
        h = np.sqrt(L2**2 - a**2)
        p3_p1 = (p3 - p1) / d
        p_mid = p1 + a * p3_p1
        p2 = p_mid + h * np.array([-p3_p1[1], p3_p1[0]])
        
        # End effector (extension of coupler link)
        eff = p1 + 1.5 * (p2 - p1)
        
        frames_p1.append(p1)
        frames_p2.append(p2)
        frames_eff.append(eff)
        
    frames_eff = np.array(frames_eff)
    
    # --- Jupyter Interactive Plot ---
    fig, ax = plt.subplots(figsize=(8, 6))
    
    def update_plot(frame=0):
        ax.clear()
        ax.set_title("Kinematic Linkage Assembly: Frame {}".format(frame), fontweight='bold')
        ax.set_xlim(-3, 10)
        ax.set_ylim(-3, 8)
        ax.set_aspect('equal')
        ax.grid(True, linestyle='--', alpha=0.6)
        
        # Fetch current points
        p1 = frames_p1[frame]
        p2 = frames_p2[frame]
        
        # Plot full path of end effector (tracer)
        ax.plot(frames_eff[:, 0], frames_eff[:, 1], color='orange', alpha=0.3, label='Target Trajectory')
        # Plot tracer up to current frame
        ax.plot(frames_eff[:frame, 0], frames_eff[:frame, 1], color='red', linewidth=2)
        
        # Plot Linkages (Bars)
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], 'k-', linewidth=4, label='Motor Crank')
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b-', linewidth=3, label='Coupler Link')
        ax.plot([p3[0], p2[0]], [p3[1], p2[1]], 'g-', linewidth=3, label='Rocker Link')
        
        # Plot End Effector Extension
        ax.plot([p1[0], frames_eff[frame, 0]], [p1[1], frames_eff[frame, 1]], 'r--', linewidth=2)
        
        # Plot Joints (Pins)
        joints_x = [p0[0], p1[0], p2[0], p3[0], frames_eff[frame, 0]]
        joints_y = [p0[1], p1[1], p2[1], p3[1], frames_eff[frame, 1]]
        ax.scatter(joints_x, joints_y, color='black', s=80, zorder=5)
        ax.scatter([p0[0], p3[0]], [p0[1], p3[1]], color='white', s=20, zorder=6) # Ground indicator
        
        ax.legend(loc='upper right')
        plt.show()

    # Create the interactive slider widget
    interact(update_plot, frame=IntSlider(min=0, max=n_frames-1, step=1, value=0, description='Time Step:'))

# To execute the visualizer in Jupyter, simply call:
# launch_jupyter_visualizer()

ImportError: dlopen(/opt/anaconda3/lib/python3.11/site-packages/scipy/linalg/_fblas.cpython-311-darwin.so, 0x0002): Library not loaded: @rpath/liblapack.3.dylib
  Referenced from: <60482C81-1F1A-3F71-B65A-80615D2F9DDB> /opt/anaconda3/lib/python3.11/site-packages/scipy/linalg/_fblas.cpython-311-darwin.so
  Reason: tried: '/opt/anaconda3/lib/python3.11/site-packages/scipy/linalg/../../../../liblapack.3.dylib' (no such file), '/opt/anaconda3/lib/python3.11/site-packages/scipy/linalg/../../../../liblapack.3.dylib' (no such file), '/opt/anaconda3/bin/../lib/liblapack.3.dylib' (no such file), '/opt/anaconda3/bin/../lib/liblapack.3.dylib' (no such file), '/usr/local/lib/liblapack.3.dylib' (no such file), '/usr/lib/liblapack.3.dylib' (no such file, not in dyld cache)